# Item-Level Difficulty Correlation: VLMs vs. Children

**Research question:** Do Vision-Language Models find the same items difficult as children?

We correlate per-item VLM accuracy with Item Response Theory (IRT) difficulty parameters
estimated from children's responses. A significant negative correlation means that items
that are harder for children (higher IRT difficulty) are also harder for models.

## Experimental Setup

- **Models:** Qwen2.5-VL-0.8B, InternVL3.5-1B, InternVL3.5-4B
- **Tasks with IRT data:** Vocab (n=126 items), TROG (n=80), EGMA-Math (n=147)
- **Prompt:** Best TF×OF from sweep (TF1_minimal × OF1_bare or OF2_json)
- **Metric:** Point-biserial correlation (r_pb) between binary item correctness and continuous IRT difficulty
- **Mental-rotation** was excluded: IRT difficulty not available for matched item_uids (0 items mapped)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

RESULTS_DIR = Path("../../results/item_difficulty")

dfs = []
for subdir in ["qwen35_0.8B", "internvl35_1B", "internvl35_4B"]:
    path = RESULTS_DIR / subdir / "item_results.csv"
    if path.exists():
        df = pd.read_csv(path)
        dfs.append(df)

items = pd.concat(dfs, ignore_index=True)
items["model_label"] = items["model"] + " " + items["model_size"]
items["irt_difficulty"] = pd.to_numeric(items["irt_difficulty"], errors="coerce")
items_irt = items.dropna(subset=["irt_difficulty"])

print(f"Total item-level observations: {len(items)}")
print(f"With IRT difficulty: {len(items_irt)}")
print(f"\nBreakdown by model × task:")
print(items_irt.groupby(["model_label", "task"]).agg(
    n=("is_correct", "count"),
    accuracy=("is_correct", "mean")
).round(3).to_string())

## 1. Correlation Summary Table

In [ ]:
summaries = []
for subdir in ["qwen35_0.8B", "internvl35_1B", "internvl35_4B"]:
    path = RESULTS_DIR / subdir / "correlation_summary.csv"
    if path.exists():
        df = pd.read_csv(path)
        df["model_label"] = subdir.replace("_", " ")
        summaries.append(df)

corr_df = pd.concat(summaries, ignore_index=True)

def sig_stars(p):
    if p < 0.001: return "***"
    if p < 0.01: return "**"
    if p < 0.05: return "*"
    if p < 0.1: return "†"
    return ""

corr_df["sig"] = corr_df["p_pointbiserial"].apply(sig_stars)
corr_df["r_pb_str"] = corr_df.apply(
    lambda r: f"{r['r_pointbiserial']:+.3f}{r['sig']}", axis=1)

display_cols = ["model_label", "task", "n_items", "accuracy", "r_pb_str", 
                "p_pointbiserial", "rho_spearman", "p_spearman"]
print(corr_df[display_cols].to_string(index=False))

## 2. Item Accuracy vs. IRT Difficulty — Scatter Plots

Each point is one item. We bin items by IRT difficulty quintile and show mean accuracy
with a logistic regression trend line. The negative slope confirms that harder items
(for children) are also harder for VLMs.

In [ ]:
tasks_with_irt = ["vocab", "trog", "egma-math"]
models = ["qwen35 0.8B", "internvl35 1B", "internvl35 4B"]
model_colors = {"qwen35 0.8B": "#e74c3c", "internvl35 1B": "#3498db", "internvl35 4B": "#2ecc71"}

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, task in zip(axes, tasks_with_irt):
    task_data = items_irt[items_irt["task"] == task]
    
    for model_label in models:
        md = task_data[task_data["model_label"] == model_label]
        if len(md) == 0:
            continue
        
        # Bin items into difficulty quintiles for visualization
        md = md.copy()
        md["diff_bin"] = pd.qcut(md["irt_difficulty"], q=5, duplicates="drop")
        binned = md.groupby("diff_bin", observed=True).agg(
            mean_diff=("irt_difficulty", "mean"),
            mean_acc=("is_correct", "mean"),
            n=("is_correct", "count")
        ).reset_index()
        
        r_pb = corr_df[(corr_df["task"] == task) & 
                       (corr_df["model_label"] == model_label.replace(" ", "_"))]
        r_str = ""
        if len(r_pb) > 0:
            row = r_pb.iloc[0]
            r_str = f" (r={row['r_pointbiserial']:+.2f}{sig_stars(row['p_pointbiserial'])})"
        
        ax.plot(binned["mean_diff"], binned["mean_acc"], 
                "o-", color=model_colors[model_label], 
                label=f"{model_label}{r_str}",
                markersize=6, linewidth=2)
    
    ax.set_xlabel("IRT Difficulty (higher = harder for children)", fontsize=11)
    ax.set_title(task.replace("-", " ").title(), fontsize=13, fontweight="bold")
    ax.axhline(y=0.25, color="gray", linestyle="--", alpha=0.5, label="Chance (4-AFC)")
    ax.legend(fontsize=8, loc="upper right")
    ax.set_ylim(-0.05, 1.05)
    ax.grid(alpha=0.3)

axes[0].set_ylabel("VLM Accuracy (binned by difficulty quintile)", fontsize=11)
fig.suptitle("VLM Accuracy vs. Child IRT Difficulty (Item-Level)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../../results/item_difficulty/accuracy_vs_irt_difficulty.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to results/item_difficulty/accuracy_vs_irt_difficulty.png")

## 3. Model Size Effect on Correlation Strength

How does model capacity affect alignment with children's difficulty ordering?

In [ ]:
size_order = {"qwen35 0.8B": 0.8, "internvl35 1B": 1.0, "internvl35 4B": 4.0}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: accuracy vs model size, by task
ax = axes[0]
task_markers = {"vocab": "o", "trog": "s", "egma-math": "D"}
task_colors = {"vocab": "#9b59b6", "trog": "#e67e22", "egma-math": "#1abc9c"}

for task in tasks_with_irt:
    task_corr = corr_df[corr_df["task"] == task].copy()
    task_corr["size_B"] = task_corr["model_label"].map(
        {k.replace(" ", "_"): v for k, v in size_order.items()})
    task_corr = task_corr.dropna(subset=["size_B"]).sort_values("size_B")
    ax.plot(task_corr["size_B"], task_corr["accuracy"], 
            f"{task_markers[task]}-", color=task_colors[task], 
            label=task, markersize=8, linewidth=2)

ax.set_xlabel("Model Size (B parameters)", fontsize=11)
ax.set_ylabel("Accuracy", fontsize=11)
ax.set_title("Accuracy vs. Model Size", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.set_xscale("log")
ax.set_xticks([0.8, 1.0, 4.0])
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.grid(alpha=0.3)

# Right: correlation strength vs model size
ax = axes[1]
for task in tasks_with_irt:
    task_corr = corr_df[corr_df["task"] == task].copy()
    task_corr["size_B"] = task_corr["model_label"].map(
        {k.replace(" ", "_"): v for k, v in size_order.items()})
    task_corr = task_corr.dropna(subset=["size_B"]).sort_values("size_B")
    
    for _, row in task_corr.iterrows():
        marker = "*" if row["p_pointbiserial"] < 0.05 else task_markers[task]
        ms = 15 if row["p_pointbiserial"] < 0.05 else 8
        ax.plot(row["size_B"], row["r_pointbiserial"], 
                marker, color=task_colors[task], markersize=ms)
    
    ax.plot(task_corr["size_B"], task_corr["r_pointbiserial"],
            "-", color=task_colors[task], linewidth=2, label=task, alpha=0.7)

ax.axhline(y=0, color="gray", linestyle="-", alpha=0.3)
ax.set_xlabel("Model Size (B parameters)", fontsize=11)
ax.set_ylabel("Point-Biserial r (VLM acc. vs. IRT difficulty)", fontsize=11)
ax.set_title("Correlation Strength vs. Model Size", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.set_xscale("log")
ax.set_xticks([0.8, 1.0, 4.0])
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.grid(alpha=0.3)
ax.annotate("★ = p < 0.05", xy=(0.02, 0.02), xycoords="axes fraction", fontsize=9, color="gray")

fig.suptitle("Model Capacity and Alignment with Child Difficulty", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../../results/item_difficulty/model_size_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Per-Item Difficulty Profile — Heatmap

Visualize which items each model gets right/wrong, sorted by IRT difficulty.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, task in zip(axes, tasks_with_irt):
    task_data = items_irt[items_irt["task"] == task].copy()
    
    # Pivot: items (rows, sorted by difficulty) × models (columns)
    pivot = task_data.pivot_table(
        index="item_uid", columns="model_label", 
        values="is_correct", aggfunc="first")
    
    # Add difficulty and sort
    diff_map = task_data.drop_duplicates("item_uid").set_index("item_uid")["irt_difficulty"]
    pivot["irt_diff"] = pivot.index.map(diff_map)
    pivot = pivot.sort_values("irt_diff")
    irt_vals = pivot["irt_diff"].values
    pivot = pivot.drop(columns=["irt_diff"])
    
    # Reorder columns by model size
    col_order = [c for c in ["qwen35 0.8B", "internvl35 1B", "internvl35 4B"] if c in pivot.columns]
    pivot = pivot[col_order]
    
    im = ax.imshow(pivot.values, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1,
                   interpolation="nearest")
    ax.set_xticks(range(len(col_order)))
    ax.set_xticklabels([c.split()[-1] for c in col_order], fontsize=10)
    ax.set_xlabel("Model", fontsize=11)
    
    # Y-axis: show difficulty range
    n_items = len(pivot)
    tick_positions = [0, n_items//4, n_items//2, 3*n_items//4, n_items-1]
    tick_labels = [f"{irt_vals[i]:.1f}" for i in tick_positions]
    ax.set_yticks(tick_positions)
    ax.set_yticklabels(tick_labels, fontsize=9)
    ax.set_ylabel("IRT Difficulty →" if ax == axes[0] else "", fontsize=11)
    ax.set_title(f"{task.replace('-', ' ').title()} (n={n_items})", fontsize=13, fontweight="bold")

cbar = fig.colorbar(im, ax=axes, shrink=0.6, label="Correct (1=green, 0=red)")
fig.suptitle("Per-Item Correctness Heatmap (sorted by IRT difficulty)", 
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../../results/item_difficulty/item_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Binned Difficulty Analysis — Error Rate by Quintile

Group items into difficulty quintiles and compare error rates across models.
This provides a clearer view than individual-item scatter.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, task in zip(axes, tasks_with_irt):
    task_data = items_irt[items_irt["task"] == task].copy()
    
    # Create quintile bins based on first model's difficulty (same items for all)
    first_model = task_data["model_label"].iloc[0]
    ref = task_data[task_data["model_label"] == first_model]
    _, bin_edges = pd.qcut(ref["irt_difficulty"], q=5, retbins=True, duplicates="drop")
    labels_q = [f"Q{i+1}" for i in range(len(bin_edges)-1)]
    task_data["quintile"] = pd.cut(task_data["irt_difficulty"], bins=bin_edges, 
                                    labels=labels_q, include_lowest=True)
    
    width = 0.25
    x = np.arange(len(labels_q))
    
    for j, model_label in enumerate(models):
        md = task_data[task_data["model_label"] == model_label]
        if len(md) == 0:
            continue
        error_rates = md.groupby("quintile", observed=True)["is_correct"].apply(
            lambda s: 1 - s.mean()).reindex(labels_q)
        ax.bar(x + j * width - width, error_rates.values, width,
               label=model_label, color=model_colors[model_label], alpha=0.85)
    
    ax.set_xlabel("Difficulty Quintile (Q1=easiest)", fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels(labels_q)
    ax.set_title(task.replace("-", " ").title(), fontsize=13, fontweight="bold")
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)

axes[0].set_ylabel("Error Rate", fontsize=11)
fig.suptitle("Error Rate by IRT Difficulty Quintile", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../../results/item_difficulty/error_by_quintile.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Key Findings

### Significant correlations (VLM accuracy vs. child IRT difficulty)

| Model | Task | n | Accuracy | r_pb | p |
|-------|------|---|----------|------|---|
| **Qwen 0.8B** | **Vocab** | 126 | 57.1% | **-0.335*** | <0.001 |
| **InternVL 4B** | **Vocab** | 126 | 88.1% | **-0.269** | 0.002 |
| **InternVL 4B** | **TROG** | 80 | 68.8% | **-0.245** | 0.028 |
| **Qwen 0.8B** | **TROG** | 80 | 41.2% | **-0.221** | 0.049 |

### Interpretation

1. **Vocab and TROG show significant item-level alignment** between VLMs and children.
   Items that are harder for children are systematically harder for VLMs.

2. **EGMA-Math shows no correlation** — math reasoning appears to rely on different
   cognitive processes than those captured by IRT difficulty from children.

3. **Model size interacts with correlation strength in a non-trivial way:**
   - InternVL 1B (lowest accuracy) shows no correlation — it performs near chance,
     so item difficulty is irrelevant.
   - Qwen 0.8B and InternVL 4B both show significant correlations, but for different
     reasons: Qwen has enough capacity to be above chance but still struggles with
     hard items; InternVL 4B is strong overall but selectively fails on the hardest items.

4. **Implication for developmental parallels:** The alignment suggests that the visual
   and linguistic features that make vocabulary and grammar items difficult for children
   (perceptual complexity, semantic ambiguity) also challenge VLMs. This supports the
   hypothesis that VLM developmental trajectories partially mirror child development.